# LoRA sur Restormer pre-entraine (deraining) -> Single Image Reflection Removal

Plan :
1. Telecharger le checkpoint officiel `deraining.pth`
2. Completer l'archi pour qu'elle matche **exactement** l'officielle (mes MDTA/GDFN ne suffisent pas, cf. plus bas)
3. Charger les poids via une table de renommage
4. Sanity check zero-shot sur SIR2 (le modele deraining fait-il deja quelque chose ?)
5. Injecter les LoRA sur les Conv 1x1
6. Geler / degeler et compter les params

## 1. Checkpoint officiel

Release `v1.0` du repo `swz30/Restormer`. Config deraining : `dim=48`, `num_blocks=[4,6,6,8]`,
`heads=[1,2,4,8]`, `ffn_expansion_factor=2.66`, `bias=False`, `LayerNorm_type='WithBias'`.
Le state_dict est sous la cle `"params"`.

In [ ]:
!wget -q -c https://github.com/swz30/Restormer/releases/download/v1.0/deraining.pth -O /content/deraining.pth
!ls -lh /content/deraining.pth

## 2. Ecarts entre mon implementation (restormer.ipynb) et l'officielle

Trois differences **structurelles** (pas juste des noms) empechent de charger le checkpoint tel quel :

| # | Mon code | Officiel |
|---|---|---|
| 1 | Le residu est **dans** MDTA/GDFN (`return x + self.proj(...)`), sans normalisation | `x = x + attn(norm1(x))` puis `x = x + ffn(norm2(x))` : le residu est dans le `TransformerBlock` et il y a **deux LayerNorm par bloc** |
| 2 | `MDTA.proj` et `GDFN` ont `bias=True` par defaut | `bias=False` partout pour le deraining -> aucune cle `.bias` dans le checkpoint |
| 3 | Seulement les blocs MDTA/GDFN | Il manque tout le U-Net : `patch_embed`, `down/up` (PixelUnshuffle/PixelShuffle), `reduce_chan_*`, `latent`, `refinement`, `output`, + **residu global** `output(x) + inp_img` |

Le reste (la math de MDTA, le gating GeLU de GDFN, les shapes) est deja **identique** a l'officiel.

Table de correspondance des noms :

| Officiel | Moi |
|---|---|
| `attn.temperature` | `attn.alpha` |
| `attn.qkv` | `attn.qkv_p` |
| `attn.qkv_dwconv` | `attn.qkv_d` |
| `attn.project_out` | `attn.proj` |
| `ffn.project_in` | `ffn.conv_p` |
| `ffn.dwconv` | `ffn.conv_d` |
| `ffn.project_out` | `ffn.proj` |
| `norm{1,2}.body.{weight,bias}` | `norm{1,2}.{weight,bias}` |

> Le residu global `output(x) + inp_img` veut dire que Restormer apprend **deja** une difference.
> C'est exactement la parametrisation que j'avais trouvee la meilleure dans fumo.ipynb : rien a changer,
> il suffit de poser `B_hat = I - T_hat`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class LayerNorm2d(nn.Module):
    """WithBias_LayerNorm de Restormer : LayerNorm sur l'axe canal, eps=1e-5.

    Attention : la variante 'BiasFree' (checkpoints de denoising) ne soustrait PAS
    la moyenne -> x / sqrt(var + eps) * weight. Ici on est en WithBias (deraining).
    """
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)                                        # (B,H,W,C)
        x = F.layer_norm(x, x.shape[-1:], self.weight, self.bias, 1e-5)
        return x.permute(0, 3, 1, 2).contiguous()


class MDTA(nn.Module):
    """Idem restormer.ipynb, mais SANS le residu interne et avec bias configurable."""
    def __init__(self, dim, num_heads, bias=False):
        super().__init__()
        assert dim % num_heads == 0, "C doit etre divisible par num_heads"
        self.num_heads = num_heads
        self.alpha = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv_p = nn.Conv2d(dim, 3 * dim, kernel_size=1, bias=bias)
        self.qkv_d = nn.Conv2d(3 * dim, 3 * dim, kernel_size=3, padding=1, groups=3 * dim, bias=bias)
        self.proj = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv_d(self.qkv_p(x)).chunk(3, dim=1)
        q, k, v = (t.reshape(B, self.num_heads, C // self.num_heads, H * W) for t in (q, k, v))
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        attn = ((q @ k.transpose(-2, -1)) * self.alpha).softmax(-1)
        out = (attn @ v).reshape(B, C, H, W)
        return self.proj(out)


class GDFN(nn.Module):
    """Idem restormer.ipynb, sans le residu interne."""
    def __init__(self, dim, lam=2.66, bias=False):
        super().__init__()
        hidden = int(dim * lam)
        self.conv_p = nn.Conv2d(dim, 2 * hidden, kernel_size=1, bias=bias)
        self.conv_d = nn.Conv2d(2 * hidden, 2 * hidden, kernel_size=3, padding=1,
                                groups=2 * hidden, bias=bias)
        self.proj = nn.Conv2d(hidden, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x1, x2 = self.conv_d(self.conv_p(x)).chunk(2, dim=1)
        return self.proj(F.gelu(x1) * x2)


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, lam=2.66, bias=False):
        super().__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn = MDTA(dim, num_heads, bias)
        self.norm2 = LayerNorm2d(dim)
        self.ffn = GDFN(dim, lam, bias)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_c=3, dim=48, bias=False):
        super().__init__()
        self.proj = nn.Conv2d(in_c, dim, kernel_size=3, padding=1, bias=bias)

    def forward(self, x):
        return self.proj(x)


class Downsample(nn.Module):
    """C -> 2C, H,W -> H/2,W/2   (bias=False en dur dans l'officiel)"""
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat // 2, kernel_size=3, padding=1, bias=False),
            nn.PixelUnshuffle(2))

    def forward(self, x):
        return self.body(x)


class Upsample(nn.Module):
    """C -> C/2, H,W -> 2H,2W"""
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat * 2, kernel_size=3, padding=1, bias=False),
            nn.PixelShuffle(2))

    def forward(self, x):
        return self.body(x)


class Restormer(nn.Module):
    def __init__(self, dim=48, num_blocks=(4, 6, 6, 8), num_refinement_blocks=4,
                 heads=(1, 2, 4, 8), lam=2.66, bias=False):
        super().__init__()
        blk = lambda d, h, n: nn.Sequential(*[TransformerBlock(d, h, lam, bias) for _ in range(n)])

        self.patch_embed = OverlapPatchEmbed(3, dim, bias)
        self.encoder_level1 = blk(dim, heads[0], num_blocks[0])
        self.down1_2 = Downsample(dim)
        self.encoder_level2 = blk(dim * 2, heads[1], num_blocks[1])
        self.down2_3 = Downsample(dim * 2)
        self.encoder_level3 = blk(dim * 4, heads[2], num_blocks[2])
        self.down3_4 = Downsample(dim * 4)
        self.latent = blk(dim * 8, heads[3], num_blocks[3])

        self.up4_3 = Upsample(dim * 8)
        self.reduce_chan_level3 = nn.Conv2d(dim * 8, dim * 4, kernel_size=1, bias=bias)
        self.decoder_level3 = blk(dim * 4, heads[2], num_blocks[2])
        self.up3_2 = Upsample(dim * 4)
        self.reduce_chan_level2 = nn.Conv2d(dim * 4, dim * 2, kernel_size=1, bias=bias)
        self.decoder_level2 = blk(dim * 2, heads[1], num_blocks[1])
        self.up2_1 = Upsample(dim * 2)
        self.decoder_level1 = blk(dim * 2, heads[0], num_blocks[0])   # pas de reduce ici
        self.refinement = blk(dim * 2, heads[0], num_refinement_blocks)
        self.output = nn.Conv2d(dim * 2, 3, kernel_size=3, padding=1, bias=bias)

    def forward(self, inp_img):
        e1 = self.encoder_level1(self.patch_embed(inp_img))
        e2 = self.encoder_level2(self.down1_2(e1))
        e3 = self.encoder_level3(self.down2_3(e2))
        lat = self.latent(self.down3_4(e3))

        d3 = self.decoder_level3(self.reduce_chan_level3(torch.cat([self.up4_3(lat), e3], 1)))
        d2 = self.decoder_level2(self.reduce_chan_level2(torch.cat([self.up3_2(d3), e2], 1)))
        d1 = self.decoder_level1(torch.cat([self.up2_1(d2), e1], 1))
        d1 = self.refinement(d1)
        return self.output(d1) + inp_img          # residu global

## 3. Chargement du checkpoint

`strict=False` d'abord pour lire `missing`/`unexpected` si ca coince, puis assert que les deux sont vides.
Si `unexpected` contient des cles en `.bias`, c'est que j'ai laisse un `bias=True` quelque part.

In [ ]:
RENAME = [
    (".norm1.body.",       ".norm1."),
    (".norm2.body.",       ".norm2."),
    (".attn.temperature",  ".attn.alpha"),
    (".attn.qkv.",         ".attn.qkv_p."),
    (".attn.qkv_dwconv.",  ".attn.qkv_d."),
    (".attn.project_out.", ".attn.proj."),
    (".ffn.project_in.",   ".ffn.conv_p."),
    (".ffn.dwconv.",       ".ffn.conv_d."),
    (".ffn.project_out.",  ".ffn.proj."),
]


def remap(sd):
    out = {}
    for k, v in sd.items():
        for src, dst in RENAME:
            k = k.replace(src, dst)
        out[k] = v
    return out


raw = torch.load("/content/deraining.pth", map_location="cpu", weights_only=True)["params"]
model = Restormer()
missing, unexpected = model.load_state_dict(remap(raw), strict=False)

print(f"missing   : {len(missing)}   {missing[:5]}")
print(f"unexpected: {len(unexpected)} {unexpected[:5]}")
assert not missing and not unexpected, "l'archi ne matche pas le checkpoint"

model = model.to(device).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M params")

## 4. Sanity check zero-shot sur SIR2

Le modele deraining n'a jamais vu de reflet. La question : est-ce que le prior "enlever un calque
superpose" transfere un peu, ou est-ce qu'il ne fait rien ?

**Important** : Restormer a ete entraine sur des entrees dans **[0,1]**, pas [-1,1].
Donc `i_t_b(i, to01=True)`.

400 et 536 sont deja divisibles par 8, donc pas besoin de padder pour l'eval pleine resolution.

In [ ]:
# --- Drive + extraction SIR2 (idem fumo.ipynb). Idempotent : ne refait rien si deja extrait.
import os, zipfile

base = "/content/data/SIR2"

if not os.path.isdir(os.path.join(base, "SolidObject")):
    from google.colab import drive
    drive.mount("/content/drive")

    with zipfile.ZipFile("/content/drive/MyDrive/hanyang/SIR2.zip") as z:
        z.extractall(base)

    inner = {
        "SolidObjectDataset.zip": "SolidObject",
        "Postcard Dataset.zip": "Postcard",
        "Wildscene.zip": "Wild",
    }
    for z, out in inner.items():
        with zipfile.ZipFile(os.path.join(base, z)) as zf:
            zf.extractall(os.path.join(base, out))
        print(f"{z} -> {out}/ OK")
else:
    print("SIR2 deja extrait")

print(sorted(os.listdir(base)))

In [ ]:
# --- reprise de fumo.ipynb (build_index / SirDataset / transforms / i_t_b / psnr) ---
import os
from torch.utils.data import Dataset
from torchvision.transforms import v2
from torchvision.io import decode_image

base = "/content/data/SIR2"


def det_role(s):
    if "r." in s or "-r" in s:
        return "r"
    elif "m." in s or "-m" in s:
        return "m"
    return "g"


def build_index(subset_root):
    dico = []
    for dirpath, dirnames, filenames in os.walk(subset_root):
        if len(filenames) == 3 and filenames[0][-3:] in ("jpg", "png"):
            dico.append({det_role(i): os.path.join(dirpath, i) for i in filenames})
    return dico


class SirDataset(Dataset):
    """ds[i] -> [R, I, T] uint8 [3,400,536]"""
    def __init__(self, base):
        self.index = build_index(base)
        self.crop = v2.CenterCrop((400, 536))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        d = self.index[i]
        r, m, g = decode_image(d["r"]), decode_image(d["m"]), decode_image(d["g"])
        return [self.crop(r), self.crop(m), self.crop(g)]


ds = SirDataset(base)
to_float = v2.ToDtype(torch.float32, scale=True)      # -> [0,1] directement


def i_t_b01(i):
    """I, T, B du sample i, dans [0,1], shape [1,3,H,W], sur device."""
    R, I, T = ds[i]
    f = lambda x: to_float(x).unsqueeze(0).to(device)
    return f(I), f(T), f(R)


def psnr(pred, target, max_val=1.0):
    mse = torch.mean((pred.float() - target.float()) ** 2)
    return (10 * torch.log10(max_val ** 2 / mse)).item()


assert len(ds) > 0, f"index vide : {base} ne contient aucune scene (dossier de 3 images). Contenu : {os.listdir(base)[:5]}"
print(f"{len(ds)} triplets")


In [ ]:
# Baseline identite vs Restormer-deraining zero-shot, sur un echantillon
import random

idx = random.sample(range(len(ds)), min(30, len(ds)))
p_id, p_zs = [], []

with torch.no_grad():
    for i in idx:
        I, T, B = i_t_b01(i)
        That = model(I).clamp(0, 1)
        p_id.append(psnr(I, T))
        p_zs.append(psnr(That, T))

print(f"baseline identite (T_hat = I) : {sum(p_id)/len(p_id):.2f} dB")
print(f"deraining zero-shot           : {sum(p_zs)/len(p_zs):.2f} dB")

In [ ]:
# Visu qualitative
i = 250
with torch.no_grad():
    I, T, B = i_t_b01(i)
    That = model(I).clamp(0, 1)

show = lambda x, t: (plt.imshow(x[0].permute(1, 2, 0).cpu().numpy()), plt.title(t), plt.axis("off"))
plt.figure(figsize=(16, 4))
for j, (x, t) in enumerate([(I, "I (mixture)"), (That, "T_hat zero-shot"),
                            (T, "T (GT)"), ((I - That + 0.5).clamp(0, 1), "B_hat = I - T_hat")]):
    plt.subplot(1, 4, j + 1); show(x, t)
plt.show()

## 5. LoRA

**Ou ?** Toutes les couches interessantes de Restormer sont des **Conv2d 1x1**, donc strictement
equivalentes a des `Linear` appliques par pixel : une conv 1x1 de poids `W` de shape `(out, in, 1, 1)`
est le lineaire `y[:, p] = W_2d @ x[:, p]`. Une LoRA de rang `r` s'ecrit donc

$$ y = Wx + \frac{\alpha}{r} B A x, \qquad A \in \mathbb{R}^{r \times in},\ B \in \mathbb{R}^{out \times r} $$

et se realise avec **deux convs 1x1 enchainees** (`in -> r -> out`). `B` initialise a **zero** :
a l'init le modele est exactement le pre-entraine, donc pas de choc au premier step.

**Cibles** : `attn.qkv_p`, `attn.proj`, `ffn.conv_p`, `ffn.proj`.

**Pas de LoRA sur** :
- les depthwise 3x3 (`qkv_d`, `conv_d`) : `groups=C`, le poids est deja de shape `(C,1,3,3)`,
  il n'y a pas de matrice `out x in` a factoriser, et c'est ~1% des params
- `down*/up*` (3x3), `patch_embed`, `output` : ce sont des 3x3, et surtout le head `output` on
  preferera le degeler entierement (c'est lui qui doit changer de tache)

In [ ]:
class LoRAConv1x1(nn.Module):
    """Enveloppe une Conv2d 1x1 gelee : y = W x + (alpha/r) * B A x, avec B = 0 a l'init."""
    def __init__(self, base: nn.Conv2d, r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert base.kernel_size == (1, 1) and base.groups == 1
        self.base = base
        self.base.requires_grad_(False)
        self.a = nn.Conv2d(base.in_channels, r, kernel_size=1, bias=False)
        self.b = nn.Conv2d(r, base.out_channels, kernel_size=1, bias=False)
        nn.init.kaiming_uniform_(self.a.weight, a=5 ** 0.5)
        nn.init.zeros_(self.b.weight)
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + self.scale * self.b(self.a(self.drop(x)))

    @torch.no_grad()
    def merge(self):
        """Replie la LoRA dans la conv de base -> plus aucun surcout a l'inference."""
        dw = (self.b.weight.flatten(1) @ self.a.weight.flatten(1)) * self.scale
        self.base.weight += dw.view_as(self.base.weight)
        return self.base


TARGETS = ("attn.qkv_p", "attn.proj", "ffn.conv_p", "ffn.proj")


def inject_lora(model, r=8, alpha=16, dropout=0.0, targets=TARGETS):
    n = 0
    for name, module in list(model.named_modules()):
        for cname, child in list(module.named_children()):
            full = f"{name}.{cname}" if name else cname
            if isinstance(child, nn.Conv2d) and any(full.endswith(t) for t in targets):
                setattr(module, cname, LoRAConv1x1(child, r, alpha, dropout))
                n += 1
    return n

In [ ]:
R, ALPHA = 8, 16

model.requires_grad_(False)                       # 1. tout geler
n = inject_lora(model, r=R, alpha=ALPHA)          # 2. les params LoRA naissent trainable
model = model.to(device)

# 3. on degele aussi : les LayerNorm (tres cheap, tres efficace en transfert),
#    la temperature des attentions, et le head `output`
for name, p in model.named_parameters():
    if name.startswith("output.") or ".norm1." in name or ".norm2." in name or name.endswith(".alpha"):
        p.requires_grad_(True)

trainable = [p for p in model.parameters() if p.requires_grad]
total = sum(p.numel() for p in model.parameters())
print(f"{n} convs 1x1 wrappees")
print(f"trainable : {sum(p.numel() for p in trainable)/1e6:.2f} M / {total/1e6:.1f} M "
      f"({100*sum(p.numel() for p in trainable)/total:.1f} %)")

In [ ]:
# Verif : a l'init, la sortie doit etre IDENTIQUE au pre-entraine (B = 0)
model.eval()
with torch.no_grad():
    I, T, B = i_t_b01(250)
    print("delta vs zero-shot :", (model(I).clamp(0, 1) - That).abs().max().item())   # ~0

## 6. Ce qu'il me reste a faire

- [ ] **Split par scene, pas par image.** SIR2 SolidObject = ~20 objets photographies avec plusieurs
      epaisseurs de verre / ouvertures : la meme scene revient plusieurs fois. Un split aleatoire sur
      `range(len(ds))` fuit massivement. Grouper par `os.path.dirname(dirpath)` (le dossier parent de
      la scene) et splitter sur les groupes. Idealement garder `Wild` entier en test.
- [ ] **Dataset d'entrainement** : crops aleatoires 256x256 sur le triplet (meme crop pour I/T/B),
      flips h/v, en **[0,1]**. Batch 2 pour commencer, monter si la VRAM suit.
- [ ] **Boucle** : AdamW sur `trainable` uniquement, `lr=2e-4`, cosine, pas de weight decay sur les
      LoRA. `torch.autocast("cuda", dtype=torch.float16)` + `GradScaler` (le T4 est Turing : **pas de
      bf16**). Clip grad a 1.0.
- [ ] **Loss** : `L1(T_hat, T)` + `0.5 * L1(grad(T_hat), grad(T))` avec mon `image_gradients` de
      fumo.ipynb. Optionnel : exclusion loss sur `B_hat = I - T_hat` (les gradients de T et de B ne
      doivent pas se superposer).
- [ ] **Eval** : PSNR/SSIM pleine resolution sur le split test, **toujours affiche a cote de la
      baseline identite**. Si je ne bats pas `psnr(I, T)`, le modele ne fait rien.
- [ ] Sauver **uniquement** les params trainable (`{k: v for k, v in model.state_dict().items() if ...}`)
      -> quelques Mo, pas 100.

Idees si ca marche :
- comparer avec le checkpoint `motion_deblurring.pth` comme point de depart (le reflet est souvent floute)
- balayer `r` dans {4, 8, 16, 32} : a 400 images le rang optimal est probablement petit
- entrainer sur des mixtures synthetiques (recette Zhang CVPR18) et garder SIR2 100% en test